In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
def setup_chronos():
    """Load Chronos model"""
    try:
        from chronos import ChronosPipeline
        
        print("Loading Chronos model...")
        pipeline = ChronosPipeline.from_pretrained(
            "amazon/chronos-t5-small",
            device_map="cpu",
            torch_dtype=torch.float32,
        )
        print("✅ Chronos model loaded successfully!")
        return pipeline
        
    except Exception as e:
        print(f"❌ Error loading Chronos: {e}")
        return None

In [3]:
def predict_chronos(pipeline, data, prediction_length=1):
    """Make prediction with Chronos"""
    try:
        # Convert to tensor
        if isinstance(data, np.ndarray):
            context = torch.tensor(data, dtype=torch.float32)
        else:
            context = torch.tensor(data.values, dtype=torch.float32)
        
        # Flatten if needed
        if len(context.shape) > 1:
            context = context.flatten()
            
        # Predict
        forecast = pipeline.predict(
            context=context,
            prediction_length=prediction_length,
            num_samples=20,
        )
        
        return forecast.median(dim=0).values.numpy()
        
    except Exception as e:
        print(f"❌ Prediction error: {e}")
        return None

In [4]:
def evaluate_chronos(pipeline, data, target_column, window_size=24, test_size=100):
    """
    Evaluate Chronos model
    
    Args:
        pipeline: Chronos model
        data: DataFrame with time series data
        target_column: name of target column
        window_size: context window size (default 24)
        test_size: number of predictions to make (default 100)
    """
    print(f"\nEvaluating Chronos model...")
    print(f"Window size: {window_size}, Test predictions: {test_size}")
    
    # Get target values
    target_values = data[target_column].values
    
    predictions = []
    actuals = []
    
    # Rolling prediction
    for i in range(test_size):
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{test_size}")
        
        # Get context window
        start_idx = len(target_values) - test_size - window_size + i
        context = target_values[start_idx:start_idx + window_size]
        
        # Actual next value
        actual = target_values[start_idx + window_size]
        
        # Predict next value
        pred = predict_chronos(pipeline, context, prediction_length=1)
        
        if pred is not None and len(pred) > 0:
            predictions.append(pred[0])
        else:
            predictions.append(context[-1])  # Fallback
        
        actuals.append(actual)
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Calculate metrics
    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    
    return mse, rmse, mae, r2, predictions, actuals

In [5]:
def save_chronos_results(mse, rmse, mae, r2, predictions, actuals, save_dir="chronos_results", additional_info=None):
    """
    Chỉ lưu kết quả đánh giá và metrics - KHÔNG lưu model pipeline
    """
    import os
    import json
    import pickle
    import numpy as np
    from datetime import datetime
    
    # Create save directory
    os.makedirs(save_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    try:
        # Đảm bảo predictions và actuals là 1D arrays
        predictions_1d = np.array(predictions).flatten()
        actuals_1d = np.array(actuals).flatten()
        residuals_1d = actuals_1d - predictions_1d
        
        # Lưu results và metrics
        results = {
            "timestamp": timestamp,
            "model_info": {
                "model_name": "amazon/chronos-t5-small",
                "model_type": "chronos_pretrained_hydrology",
                "note": "Model pipeline not saved - only evaluation results"
            },
            "metrics": {
                "mse": float(mse),
                "rmse": float(rmse),
                "mae": float(mae),
                "r2": float(r2),
                "num_predictions": len(predictions_1d)
            },
            "predictions": predictions_1d.tolist(),
            "actuals": actuals_1d.tolist(),
            "residuals": residuals_1d.tolist(),
            "additional_info": additional_info or {}
        }
        
        # Save as JSON
        json_path = os.path.join(save_dir, f"evaluation_results_{timestamp}.json")
        with open(json_path, 'w') as f:
            json.dump(results, f, indent=2)
        
        # Save as pickle for easy loading
        pickle_path = os.path.join(save_dir, f"evaluation_data_{timestamp}.pkl")
        with open(pickle_path, 'wb') as f:
            pickle.dump(results, f)
        
        # Save predictions as CSV for easy analysis
        csv_path = os.path.join(save_dir, f"predictions_{timestamp}.csv")
        import pandas as pd
        pred_df = pd.DataFrame({
            'actual': actuals_1d,
            'predicted': predictions_1d,
            'residual': residuals_1d,
            'absolute_error': np.abs(residuals_1d)
        })
        pred_df.to_csv(csv_path, index=False)
        
        # Create summary report
        summary = f"""# Chronos Hydrology Evaluation Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Model Information
- Model: Amazon Chronos T5-Small (Pre-trained)
- Task: An Khê Inflow Prediction (luu_luong_den_ho)
- Input Features: luu_luong_den_ho, luu_luong_xa, hokanak135928, hoankhe135931, trammuavinhthuan82897

## Performance Metrics
- **MSE**: {mse:.6f}
- **RMSE**: {rmse:.6f} 
- **MAE**: {mae:.6f}
- **R²**: {r2:.6f}
- **Predictions**: {len(predictions_1d)} samples

## Statistical Summary
- Mean Actual: {np.mean(actuals_1d):.6f}
- Mean Predicted: {np.mean(predictions_1d):.6f}
- Mean Absolute Error: {np.mean(np.abs(residuals_1d)):.6f}
- Std Residuals: {np.std(residuals_1d):.6f}

## Data Information
"""
        if additional_info:
            for key, value in additional_info.items():
                summary += f"- **{key}**: {value}\n"
        
        summary += f"""
## Files Generated
- Evaluation Results: `{json_path}`
- Data Backup: `{pickle_path}`
- Predictions CSV: `{csv_path}`
- This Report: `evaluation_summary_{timestamp}.md`

## Note
⚠️ **Model pipeline not saved** due to serialization limitations.
To reuse the model, run the setup_chronos() function again:

```python
pipeline = setup_chronos()
prediction = predict_chronos(pipeline, your_data)
```

## Results Analysis
The model achieved an R² of {r2:.4f}, indicating {"good" if r2 > 0.7 else "moderate" if r2 > 0.5 else "poor"} predictive performance.
MAE of {mae:.6f} suggests the average prediction error is {mae:.6f} units.
"""
        
        summary_path = os.path.join(save_dir, f"evaluation_summary_{timestamp}.md")
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write(summary)
        
        print(f"\n✅ Evaluation results saved successfully!")
        print(f"📁 Directory: {save_dir}")
        print(f"📊 Results JSON: {json_path}")
        print(f"📈 Predictions CSV: {csv_path}")
        print(f"📄 Summary: {summary_path}")
        print(f"\n💡 Note: Model pipeline not saved - use setup_chronos() to reload model")
        
        return save_dir, timestamp
        
    except Exception as e:
        print(f"❌ Save error: {e}")
        import traceback
        traceback.print_exc()
        return None, None

In [6]:
def load_chronos_results(save_dir="chronos_results", timestamp=None):
    """Load saved evaluation results"""
    import os
    import json
    
    try:
        # Find latest timestamp if not provided
        if timestamp is None:
            files = [f for f in os.listdir(save_dir) if f.startswith("evaluation_results_")]
            if not files:
                print("❌ No saved results found")
                return None
            timestamp = max([f.split("_")[2].split(".")[0] for f in files])
        
        # Load results
        json_path = os.path.join(save_dir, f"evaluation_results_{timestamp}.json")
        with open(json_path, 'r') as f:
            results = json.load(f)
        
        print(f"✅ Results loaded: {timestamp}")
        print(f"📊 Metrics: MSE={results['metrics']['mse']:.6f}, RMSE={results['metrics']['rmse']:.6f}")
        print(f"         MAE={results['metrics']['mae']:.6f}, R²={results['metrics']['r2']:.6f}")
        
        return results
        
    except Exception as e:
        print(f"❌ Load error: {e}")
        return None


In [7]:
def load_and_preprocess_data():
    """
    Load và preprocess 3 files dữ liệu
    """
    try:
        print("Loading data files...")
        
        # 1. Load AnKhe.csv
        ankhe_df = pd.read_csv("AnKhe.csv")
        print(f"✅ AnKhe.csv loaded: {ankhe_df.shape}")
        
        # 2. Load KaNak.csv  
        kanak_df = pd.read_csv("KaNak.csv")
        print(f"✅ KaNak.csv loaded: {kanak_df.shape}")
        
        # 3. Load SoLieuMua_AnKhe_Kanak.csv
        rain_df = pd.read_csv("SoLieuMua_AnKhe_Kanak.csv")
        print(f"✅ SoLieuMua_AnKhe_Kanak.csv loaded: {rain_df.shape}")
        
        # Preprocessing AnKhe data
        ankhe_df["datetime"] = pd.to_datetime(
            ankhe_df["gio"].astype(str) + " " + ankhe_df["ngay"], 
            format="%H %d/%m/%Y", errors='coerce'
        )
        ankhe_clean = ankhe_df[["datetime", "luu_luong_den_ho"]].dropna()
        
        # Preprocessing KaNak data  
        kanak_df["datetime"] = pd.to_datetime(
            kanak_df["gio"].astype(str) + " " + kanak_df["ngay"],
            format="%H %d/%m/%Y", errors='coerce'
        )
        kanak_clean = kanak_df[["datetime", "luu_luong_xa"]].dropna()
        
        # Preprocessing Rain data
        rain_df["datetime"] = pd.to_datetime(rain_df["datetime"], errors='coerce')
        rain_clean = rain_df[["datetime", "hokanak135928", "hoankhe135931", "trammuavinhthuan82897"]].dropna()
        
        # Merge all data
        print("Merging datasets...")
        df = pd.merge(ankhe_clean, kanak_clean, on="datetime", how="inner")
        df = pd.merge(df, rain_clean, on="datetime", how="inner")
        
        # Create combined features
        df["total_rain"] = df["hokanak135928"] + df["hoankhe135931"] + df["trammuavinhthuan82897"]
        
        # Sort by datetime
        df = df.sort_values("datetime").reset_index(drop=True)
        
        print(f"✅ Data merged successfully!")
        print(f"📊 Final dataset shape: {df.shape}")
        print(f"📅 Date range: {df['datetime'].min()} to {df['datetime'].max()}")
        
        # Display feature info
        features = ["luu_luong_den_ho", "luu_luong_xa", "hokanak135928", "hoankhe135931", "trammuavinhthuan82897", "total_rain"]
        print(f"\n📈 Features summary:")
        for feature in features:
            if feature in df.columns:
                print(f"   {feature}: mean={df[feature].mean():.3f}, std={df[feature].std():.3f}")
        
        return df
        
    except Exception as e:
        print(f"❌ Data loading failed: {e}")
        return None

In [8]:
def split_train_test(df, train_ratio=0.8):
    """
    Chia dữ liệu thành train và test
    """
    split_idx = int(len(df) * train_ratio)
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"📊 Data split:")
    print(f"   Train: {len(train_df)} samples ({train_ratio*100:.0f}%)")
    print(f"   Test:  {len(test_df)} samples ({(1-train_ratio)*100:.0f}%)")
    
    return train_df, test_df

In [9]:
def evaluate_multivariate_chronos(pipeline, train_df, test_df, target_column="luu_luong_den_ho", 
                                 feature_columns=None, window_size=24, test_size=100):
    """
    Evaluate Chronos với multivariate input
    """
    if feature_columns is None:
        feature_columns = ["luu_luong_den_ho", "luu_luong_xa", "hokanak135928", "hoankhe135931", "trammuavinhthuan82897"]
    
    print(f"\nEvaluating Chronos with multivariate input...")
    print(f"Target: {target_column}")
    print(f"Features: {feature_columns}")
    print(f"Window size: {window_size}, Test predictions: {test_size}")
    
    # Prepare multivariate data (chỉ dùng target cho Chronos - nó là univariate model)
    target_values = test_df[target_column].values
    
    # Ensure we have enough data
    if len(target_values) < window_size + test_size:
        test_size = len(target_values) - window_size
        print(f"⚠️  Adjusted test_size to {test_size} due to data constraints")
    
    predictions = []
    actuals = []
    
    print("Making predictions...")
    for i in range(test_size):
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{test_size}")
        
        # Context window từ target values
        context_start = i
        context_end = i + window_size
        context = target_values[context_start:context_end]
        
        # Actual next value
        actual = target_values[context_end]
        
        # Predict với Chronos (chỉ dùng univariate target)
        pred = predict_chronos(pipeline, context, prediction_length=1)
        
        if pred is not None and len(pred) > 0:
            predictions.append(pred[0])
        else:
            predictions.append(context[-1])  # Fallback
        
        actuals.append(actual)
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Calculate metrics
    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    
    return mse, rmse, mae, r2, predictions, actuals

In [10]:
def main():
    """
    Main function - Chronos pipeline cho dữ liệu thủy văn
    """
    print("=== CHRONOS PIPELINE FOR HYDROLOGY DATA ===\n")
    
    # 1. Load và preprocess data
    df = load_and_preprocess_data()
    if df is None:
        return
    
    # 2. Split train/test
    train_df, test_df = split_train_test(df, train_ratio=0.8)
    
    # 3. Load Chronos model
    pipeline = setup_chronos()
    if pipeline is None:
        return
    
    # 4. Define features và target
    target_column = "luu_luong_den_ho"
    feature_columns = ["luu_luong_den_ho", "luu_luong_xa", "hokanak135928", "hoankhe135931", "trammuavinhthuan82897"]
    
    # 5. Evaluate model
    mse, rmse, mae, r2, predictions, actuals = evaluate_multivariate_chronos(
        pipeline, train_df, test_df, 
        target_column=target_column,
        feature_columns=feature_columns,
        window_size=24, 
        test_size=min(200, len(test_df) - 24)
    )
    
    # 6. Print results
    print(f"\n" + "="*60)
    print("CHRONOS EVALUATION RESULTS")
    print("="*60)
    print(f"Target: {target_column} (An Khê inflow)")
    print(f"Test samples: {len(predictions)}")
    print(f"\n📊 Performance Metrics:")
    print(f"   MSE  (Mean Squared Error):     {mse:.6f}")
    print(f"   RMSE (Root Mean Squared Error): {rmse:.6f}")
    print(f"   MAE  (Mean Absolute Error):    {mae:.6f}")
    print(f"   R²   (R-squared):              {r2:.6f}")
    
    # Additional statistics
    print(f"\n📈 Prediction Statistics:")
    print(f"   Mean actual value:     {np.mean(actuals):.6f}")
    print(f"   Mean predicted value:  {np.mean(predictions):.6f}")
    print(f"   Prediction range:      [{np.min(predictions):.3f}, {np.max(predictions):.3f}]")
    print(f"   Actual range:          [{np.min(actuals):.3f}, {np.max(actuals):.3f}]")
    
    # 7. Save model với additional info
    save_info = {
        "target_column": target_column,
        "feature_columns": feature_columns,
        "train_samples": len(train_df),
        "test_samples": len(test_df),
        "window_size": 24,
        "data_range": f"{df['datetime'].min()} to {df['datetime'].max()}"
    }
    
    save_dir, timestamp = save_chronos_results(
        mse, rmse, mae, r2, predictions, actuals,  # ✅ bỏ tham số pipeline
        additional_info=save_info
    )
    
    if save_dir:
        print(f"\n🎉 Pipeline completed successfully!")
        print(f"📁 Model saved in: {save_dir}")
        print(f"🔖 Timestamp: {timestamp}")
        
        # Test loading
        print(f"\n🧪 Testing model loading...")
        loaded_results = load_chronos_results(save_dir, timestamp)
        
        if loaded_results:
            print("✅ Model loading test successful!")
            
            # Quick test prediction
            test_context = test_df[target_column].values[:24]
            test_pred = predict_chronos(pipeline, test_context, prediction_length=1)
            if test_pred is not None:
                # Convert to scalar if needed
                if isinstance(test_pred, np.ndarray):
                    pred_value = float(test_pred[0]) if len(test_pred) > 0 else float(test_pred)
                else:
                    pred_value = float(test_pred)
                print(f"🔮 Test prediction: {pred_value:.6f}")
        else:
            print("❌ Model loading test failed")
    
    return pipeline, mse, rmse, mae, r2, df


In [11]:
# Quick usage functions cho hydrology data
def quick_predict_hydrology(data, reload_model=True):
    pipeline = setup_chronos()
    if pipeline:
        return predict_chronos(pipeline, data, prediction_length=1)
    return None

def train_and_evaluate_hydrology():
    """Train và evaluate trong một function cho hydrology data"""
    # Load data
    df = load_and_preprocess_data()
    if df is None:
        return None
    
    # Split data
    train_df, test_df = split_train_test(df)
    
    # Load model
    pipeline = setup_chronos()
    if pipeline is None:
        return None
    
    # Evaluate
    mse, rmse, mae, r2, preds, actuals = evaluate_multivariate_chronos(
        pipeline, train_df, test_df, 
        target_column="luu_luong_den_ho",
        window_size=24, 
        test_size=min(200, len(test_df) - 24)
    )
    
    # Save
    save_info = {
        "target_column": "luu_luong_den_ho",
        "feature_columns": ["luu_luong_den_ho", "luu_luong_xa", "hokanak135928", "hoankhe135931", "trammuavinhthuan82897"],
        "train_samples": len(train_df),
        "test_samples": len(test_df)
    }
    
    save_chronos_results(mse, rmse, mae, r2, preds, actuals, additional_info=save_info)
    
    return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2}

In [12]:
def predict_ankhe_inflow(context_data, prediction_length=1):
    """
    Dự đoán luu_luong_den_ho của An Khê - reload model mỗi lần
    
    Args:
        context_data: list hoặc array của historical luu_luong_den_ho values
        prediction_length: số bước dự đoán (default=1)
    """
    pipeline = setup_chronos()
    if pipeline is None:
        return None
    
    prediction = predict_chronos(pipeline, context_data, prediction_length)
    
    if prediction is not None:
        print(f"🔮 Predicted An Khê inflow: {prediction}")
        
        # Load latest results if available
        try:
            results = load_chronos_results("chronos_results")
            if results:
                print(f"📊 Latest model performance: MAE={results['metrics']['mae']:.4f}, R²={results['metrics']['r2']:.4f}")
        except:
            pass
    
    return prediction

In [13]:
if __name__ == "__main__":
    # Run main pipeline
    main()
    
    # Example usage:
    # results = train_and_evaluate("my_data.csv", "my_target_column")
    # prediction = quick_predict([1.0, 2.0, 3.0, 4.0])

=== CHRONOS PIPELINE FOR HYDROLOGY DATA ===

Loading data files...
✅ AnKhe.csv loaded: (32746, 7)
✅ KaNak.csv loaded: (32763, 7)
✅ SoLieuMua_AnKhe_Kanak.csv loaded: (36514, 9)
Merging datasets...
✅ Data merged successfully!
📊 Final dataset shape: (11949, 7)
📅 Date range: 2019-10-08 00:00:00 to 2023-07-01 23:00:00

📈 Features summary:
   luu_luong_den_ho: mean=37.622, std=50.262
   luu_luong_xa: mean=22.756, std=37.219
   hokanak135928: mean=0.120, std=1.120
   hoankhe135931: mean=0.157, std=1.257
   trammuavinhthuan82897: mean=0.164, std=1.267
   total_rain: mean=0.441, std=2.512
📊 Data split:
   Train: 9559 samples (80%)
   Test:  2390 samples (20%)
Loading Chronos model...
✅ Chronos model loaded successfully!

Evaluating Chronos with multivariate input...
Target: luu_luong_den_ho
Features: ['luu_luong_den_ho', 'luu_luong_xa', 'hokanak135928', 'hoankhe135931', 'trammuavinhthuan82897']
Window size: 24, Test predictions: 200
Making predictions...
Progress: 20/200
Progress: 40/200
Progre